# Módulo 06 — Perceptron Multicamadas (`PMC` / `MLP`) e `Backpropagation`

## 1. Como Quebrar a Barreira Linear? (A Ideia das Camadas Ocultas)

No Módulo 05 vimos que **um único neurônio só traça uma única linha reta**, falhando no problema do `XOR` e na maioria dos fenômenos não lineares da Engenharia (escoamento turbulento, cinética química, resistência de materiais impressos em 3D).

A solução genial do **Perceptron Multicamadas (PMC — em inglês *Multi-Layer Perceptron*, `MLP`)** é organizar os neurônios em **estágios em cascata**:
1. **Camada de Entrada:** Recebe as variáveis medidas $x_1, x_2, \dots, x_n$.
2. **Camada(s) Oculta(s) (ou Intermediária):** Cada neurônio oculto traça uma fronteira própria, transformando o espaço geométrico.
3. **Camada de Saída:** Combina as decisões dos neurônios ocultos para produzir curvas suaves ou regiões fechadas arbitrárias (**Teorema da Aproximação Universal**).

<div align="center">
    <img src="imagens/topologia_pmc_generico.svg" alt="Arquitetura genérica de um Perceptron Multicamadas (PMC)" width="620">
</div>

*Na arquitetura genérica acima, $n$ sinais de entrada ($x_1, x_2, \dots, x_n$) atravessam sucessivas camadas ocultas ($n_1, n_2$ neurônios) até a camada de saída ($y_1, y_2, \dots, y_m$), conectadas pelas matrizes de pesos sinápticos $\mathbf{W}^{(1)}, \mathbf{W}^{(2)}, \mathbf{W}^{(3)}$.*

---

## 2. Por que Precisamos Trocar a Função Degrau pela Sigmóide?

Em uma rede com camadas internas surge um dilema: **quando a saída final $y$ erra, qual neurônio da camada oculta teve mais culpa pelo erro?**

Para descobrir isso, usamos **derivadas parciais** (a inclinação da superfície de erro). A função degrau tem derivada **zero** em toda parte (é reta horizontal!) e infinita no salto — ela é "cega" para o Cálculo Diferencial!

Por isso, no PMC substituímos o degrau por **funções contínuas e deriváveis**, principalmente a **Função Logística (Sigmóide)**:
$$g(u) = \frac{1}{1 + e^{-u}} \qquad \text{cuja derivada elegante é:} \qquad g'(u) = g(u)\cdot\big(1 - g(u)\big)$$

Repare que maravilha computacional: se você já calculou a saída $y = g(u)$, a derivada é simplesmente $y \cdot (1 - y)$!

---

## 3. Como Funciona o `Backpropagation` (Retropropagação do Erro)?

Imagine uma esfera apoiada sobre uma superfície montanhosa onde a altitude representa o **Erro Quadrático** $E = \frac{1}{2}\sum (d - y)^2$. Queremos descer até o fundo do vale (**Gradiente Descendente**).

Cada passo de aprendizado tem **duas viagens**:

### Viagem de Ida (*Forward Pass* — Propagação Direta)
Os sinais fluem da esquerda para a direita:
1. Entrada $\mathbf{x}$ multiplica os pesos $\mathbf{W}^{(1)}$ da camada oculta $\rightarrow$ passa pela sigmóide $\rightarrow$ gera as saídas ocultas $\mathbf{y}^{(1)}$.
2. Saídas ocultas $\mathbf{y}^{(1)}$ multiplicam os pesos $\mathbf{W}^{(2)}$ da camada de saída $\rightarrow$ passa pela sigmóide $\rightarrow$ gera a resposta final $\mathbf{y}^{(2)}$ e o erro $\mathbf{e} = \mathbf{d} - \mathbf{y}^{(2)}$.

### Viagem de Volta (*Backward Pass* — Regra da Cadeia)
Distribuímos a responsabilidade pelo erro de trás para frente usando a **Regra da Cadeia do Cálculo**:
1. **Gradiente local da saída ($\boldsymbol{\delta}^{(2)}$):** Erro multiplicado pela inclinação da sigmóide:
   $$\boldsymbol{\delta}^{(2)} = (\mathbf{d} - \mathbf{y}^{(2)}) \odot \big(\mathbf{y}^{(2)} \cdot (1 - \mathbf{y}^{(2)})\big)$$
2. **Gradiente local da camada oculta ($\boldsymbol{\delta}^{(1)}$):** Pega o $\boldsymbol{\delta}^{(2)}$ de volta através dos pesos $\mathbf{W}^{(2)}$ (sem a coluna do *bias*) e multiplica pela inclinação da camada oculta:
   $$\boldsymbol{\delta}^{(1)} = \big(\mathbf{y}^{(1)} \cdot (1 - \mathbf{y}^{(1)})\big) \odot \big((\mathbf{W}^{(2)}_{\text{sem bias}})^T \boldsymbol{\delta}^{(2)}\big)$$
3. **Ajuste das válvulas (Pesos) com *Momentum* ($\alpha$):**
   $$\Delta \mathbf{W}(t) = \eta \, \boldsymbol{\delta} \, \mathbf{entrada}^T + \alpha \, \Delta \mathbf{W}(t-1)$$
   > **Analogia da Inércia (*Momentum* $\alpha$):** Somar uma fração $\alpha$ (ex.: `0,8` ou `0,9`) do passo anterior dá **inércia mecânica** à esfera, ajudando-a a ganhar velocidade em ladeiras suaves e atravessar pequenos buracos (*mínimos locais*) sem ficar presa!

Vamos ver essa mágica acontecer ao vivo resolvendo o **`XOR`** que havia derrotado o Perceptron simples!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

# Funções auxiliares da ativação Logística (Sigmóide)
def sigmoide(u):
    return 1.0 / (1.0 + np.exp(-u))

def derivada_sigmoide(y):
    # Recebe 'y' já passado pela sigmóide e retorna y * (1 - y)
    return y * (1.0 - y)

# Dados do problema XOR com alvos em [0, 1] (faixa natural da função sigmóide!)
X_xor = np.array([
    [0.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [1.0, 1.0]
])
D_xor = np.array([[0.0], [1.0], [1.0], [0.0]])  # 0 quando iguais, 1 quando diferentes!

# Inicialização reprodutível de uma rede 2-4-1 (2 entradas, 4 neurônios ocultos, 1 saída)
rng = np.random.default_rng(42)
W1 = rng.uniform(-1.0, 1.0, size=(4, 3))  # 4 neurônios ocultos x (bias + 2 entradas)
W2 = rng.uniform(-1.0, 1.0, size=(1, 5))  # 1 neurônio de saída x (bias + 4 ocultos)

eta = 0.5     # Taxa de aprendizado
alpha = 0.8   # Fator de Momentum (inércia mecânica)
dW1_ant = np.zeros_like(W1)
dW2_ant = np.zeros_like(W2)
historico_mse = []

# Treinamento por 2500 épocas
for epoca in range(2500):
    erro_quadratico_soma = 0.0

    for i in range(len(X_xor)):
        # --- 1. FORWARD PASS (Ida) ---
        # Vetor coluna de entrada com x0 = -1 no topo (formato 3x1)
        x_b = np.vstack([[-1.0], X_xor[i].reshape(-1, 1)])

        # Camada oculta: potencial u1 (4x1) e ativação y1 (4x1)
        y1 = sigmoide(W1 @ x_b)

        # Camada de saída: adiciona bias -1 no topo de y1 (formato 5x1) e calcula y2 (1x1)
        y1_b = np.vstack([[-1.0], y1])
        y2 = sigmoide(W2 @ y1_b)

        # Erro instantâneo da amostra: e = desejado - previsto
        erro = D_xor[i].reshape(-1, 1) - y2
        erro_quadratico_soma += 0.5 * float(np.sum(erro ** 2))

        # --- 2. BACKWARD PASS (Volta / Regra da Cadeia) ---
        # Gradiente local na saída (1x1)
        delta2 = erro * derivada_sigmoide(y2)

        # Gradiente local na oculta (4x1): W2[:, 1:] ignora a coluna 0 (peso do bias)
        delta1 = derivada_sigmoide(y1) * (W2[:, 1:].T @ delta2)

        # --- 3. ATUALIZAÇÃO DOS PESOS COM MOMENTUM ---
        dW2 = eta * (delta2 @ y1_b.T) + alpha * dW2_ant
        W2 = W2 + dW2
        dW2_ant = dW2

        dW1 = eta * (delta1 @ x_b.T) + alpha * dW1_ant
        W1 = W1 + dW1
        dW1_ant = dW1

    historico_mse.append(erro_quadratico_soma / len(X_xor))

print(f"MSE final após 2500 épocas: {historico_mse[-1]:.6f}\n")
print("--- Prova Real nas 4 Combinações do XOR ---")
for i in range(len(X_xor)):
    x_b = np.vstack([[-1.0], X_xor[i].reshape(-1, 1)])
    y1_b = np.vstack([[-1.0], sigmoide(W1 @ x_b)])
    y_pred = float(sigmoide(W2 @ y1_b)[0, 0])
    classe = 1 if y_pred >= 0.5 else 0
    print(f"Entrada {X_xor[i]} -> Alvo: {int(D_xor[i, 0])} | Saída contínua: {y_pred:.4f} -> Classe: {classe}")

**Resultado Histórico:** O PMC cravou **100% de acerto no `XOR`** (`[0, 1, 1, 0]`), com saídas contínuas coladas em `0.01` e `0.99`! Vamos visualizar a queda do Erro Quadrático Médio (`MSE`) e a superfície curva de decisão criada pela rede:

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

# Subgráfico 1: Curva de aprendizado (queda do MSE em escala logarítmica)
ax1.plot(historico_mse, color="darkblue", linewidth=2)
ax1.set_yscale("log")
ax1.set_title("Queda do Erro Quadrático Médio (MSE)")
ax1.set_xlabel("Época")
ax1.set_ylabel("MSE (escala log)")
ax1.grid(True, which="both", linestyle="--", alpha=0.5)

# Subgráfico 2: Mapa de contorno da fronteira NÃO LINEAR aprendida no plano [0, 1] x [0, 1]
grade = np.linspace(-0.2, 1.2, 80)
G1, G2 = np.meshgrid(grade, grade)
Z = np.zeros_like(G1)
for r in range(G1.shape[0]):
    for c in range(G1.shape[1]):
        xb = np.array([[-1.0], [G1[r, c]], [G2[r, c]]])
        y1b = np.vstack([[-1.0], sigmoide(W1 @ xb)])
        Z[r, c] = float(sigmoide(W2 @ y1b)[0, 0])

mapa = ax2.contourf(G1, G2, Z, levels=20, cmap="coolwarm", alpha=0.8)
plt.colorbar(mapa, ax=ax2, label="Saída contínua y")
ax2.scatter([0, 1], [0, 1], color="navy", edgecolors="white", s=110, label="Classe 0 (iguais)")
ax2.scatter([0, 1], [1, 0], color="darkred", edgecolors="white", marker="^", s=120, label="Classe 1 (diferentes)")
ax2.set_title("Região Não Linear Criada pelo PMC no XOR")
ax2.set_xlabel("$x_1$")
ax2.set_ylabel("$x_2$")
ax2.legend(loc="upper center")

plt.tight_layout()
plt.show()

**Interpretação Geométrica:** Olhe para o mapa à direita! A camada oculta criou uma **faixa diagonal curva** vermelha no meio do plano que isola `(0, 1)` e `(1, 0)` dos cantos azuis `(0, 0)` e `(1, 1)` — algo impossível para uma única reta!

---

## 4. Boas Práticas Essenciais antes de Ir para os Projetos de Engenharia

1. **Escalonamento de Entradas (`StandardScaler` / `MinMaxScaler`):** Nunca jogue valores como `250` na função sigmóide sem normalizar! Como $e^{-250} \approx 0$, a sigmóide trava em `1.0`, onde sua derivada $y(1-y) = 1(0) = 0$ zera o aprendizado (*saturação do gradiente*).
2. **Prevenção de *Overfitting* (Sobreajuste):** Mais neurônios nem sempre é melhor! Uma rede gigante em poucos ensaios decora o ruído experimental. Sempre avaliamos a rede em um **Conjunto de Teste separado** que nunca participou do treinamento.
3. **Na Indústria (`Scikit-Learn` e `Keras`):** Depois de compreender como o motor funciona por dentro em `NumPy`, usamos `MLPClassifier` / `MLPRegressor` (do `scikit-learn`) ou `Sequential` (do `tensorflow.keras`), que já trazem otimizadores modernos como **`Adam`** e ativação **`ReLU`** ($\max(0, u)$) prontos em 5 linhas de código!

---

## 5. Exercícios Práticos de Fixação

### Exercício 1: Entendendo a Saturação da Derivada da Sigmóide
Calcule `y = sigmoide(u)` e `derivada = y * (1 - y)` para `u = 0.0`, `u = 2.0` e `u = 15.0`. Onde a derivada é máxima e o que acontece com ela quando `u = 15.0`?

In [ ]:
# Escreva seu código aqui

<details>
<summary><b>Clique aqui para ver o gabarito comentado</b></summary>

```python
import numpy as np

for u in [0.0, 2.0, 15.0]:
    y = 1.0 / (1.0 + np.exp(-u))
    deriv = y * (1.0 - y)
    print(f"u = {u:4.1f}  -->  y = {y:.6f}  |  derivada y*(1-y) = {deriv:.8f}")
# Conclusão: Em u=0 a derivada é máxima (0.25); em u=15 ela cai para ~0.0000003,
# paralisando o Backpropagation se as entradas não forem normalizadas!
```

</details>

### Exercício 2: Comparando Treino do `XOR` SEM *Momentum* (`alpha = 0.0`)
No nosso código do `XOR` usamos `alpha = 0.8`. O que acontece com o `MSE` final após as mesmas `2500` épocas se desligarmos a inércia mecânica (`alpha = 0.0`) mantendo os mesmos pesos iniciais (`seed=42`) e `eta = 0.5`? Faça o teste!

In [ ]:
# Escreva seu código aqui

<details>
<summary><b>Clique aqui para ver o gabarito comentado</b></summary>

```python
import numpy as np

rng_sem = np.random.default_rng(42)
W1_s = rng_sem.uniform(-1.0, 1.0, size=(4, 3))
W2_s = rng_sem.uniform(-1.0, 1.0, size=(1, 5))

for _ in range(2500):
    soma_e = 0.0
    for i in range(len(X_xor)):
        xb = np.vstack([[-1.0], X_xor[i].reshape(-1, 1)])
        y1 = sigmoide(W1_s @ xb)
        y1b = np.vstack([[-1.0], y1])
        y2 = sigmoide(W2_s @ y1b)
        e = D_xor[i].reshape(-1, 1) - y2
        soma_e += 0.5 * float(np.sum(e ** 2))
        d2 = e * derivada_sigmoide(y2)
        d1 = derivada_sigmoide(y1) * (W2_s[:, 1:].T @ d2)
        W2_s += 0.5 * (d2 @ y1b.T)   # alpha = 0.0 (sem inércia)
        W1_s += 0.5 * (d1 @ xb.T)

print(f"MSE após 2500 épocas SEM Momentum: {soma_e / len(X_xor):.6f}")
print(f"MSE após 2500 épocas COM Momentum: {historico_mse[-1]:.6f}")
```

</details>

### Exercício 3: Resolvendo o mesmo `XOR` em 5 Linhas com `Scikit-Learn`
Importe `from sklearn.neural_network import MLPClassifier`, instancie `clf = MLPClassifier(hidden_layer_sizes=(4,), activation='logistic', solver='lbfgs', random_state=42)`, treine com `clf.fit(X_xor, [0, 1, 1, 0])` e imprima `clf.predict(X_xor)`.

In [ ]:
# Escreva seu código aqui

<details>
<summary><b>Clique aqui para ver o gabarito comentado</b></summary>

```python
from sklearn.neural_network import MLPClassifier

# O solver 'lbfgs' (quase-Newton) converge quase instantaneamente em tabelas pequenas!
clf = MLPClassifier(hidden_layer_sizes=(4,), activation="logistic", solver="lbfgs", random_state=42)
clf.fit(X_xor, [0, 1, 1, 0])

print("Previsões do MLPClassifier no XOR:", clf.predict(X_xor))
```

</details>

---

## Próximo Passo

Parabéns! Você concluiu toda a **Trilha Base** do repositório. Agora é hora de colocar a mão na massa nos desafios práticos:
1. **[Desafios do Titanic](titanic/README.md)** (Estatística, Pandas, Gráficos, Perceptron e MLP)
2. **[Desafios da Flor de Íris](iris/README.md)** (Classificação Multiclasse)
3. **[Projetos Aplicados de Engenharia](engenharia/README.md)** (Petróleo, Ressonância Magnética, Conservantes, Séries Temporais e Impressão 3D)!